# Shared data on Kaggle: download, splits, pairs (owner: Cường; run once for the team)

Settings: **CPU is enough** (no GPU needed), **Internet on**. Optional: add a Kaggle secret `HF_TOKEN`
(a read token from huggingface.co) for faster, rate-limit-free downloads.

What this notebook produces in `/kaggle/working` (kept as the notebook version's output):

- `AnimeFaceColorization/` — the full HF mirror (~8.4 GB). Turn the output into a **private Kaggle Dataset**
  ("New Dataset" from this notebook's output) and share it with the other two accounts.
- `splits/` — `train/val/test/test_cross.txt` + `cross_ref_pairs.csv` + `shuffled_ref_pairs.csv`.
  **Download these and commit them to the repo** (fairness rule 3: generated once, never ad hoc).
- `previews/` — S / Y / R grids to check the loader by eye.

In [ ]:
REPO_URL = "https://github.com/<org>/<repo>.git"   # fill in
BRANCH = "main"
!git clone -q -b $BRANCH $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

## 1. Download the HF mirror
Skips files that are already there, so a rerun after an interruption continues.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("using HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: anonymous download (slower, may be rate-limited)")

DATA = "/kaggle/working/AnimeFaceColorization"
os.environ["ANIME_DATA_ROOT"] = DATA
!hf download ComputerVisionAnimeProject/AnimeFaceColorization --repo-type dataset --local-dir $DATA --max-workers 32
!rm -rf $DATA/.cache
!du -sh $DATA

## 2. What is in the mirror
Group = one folder. `AnimeDiffusion_<n>` holds one drawing plus two deformation-flow copies; the other
folders are moeimouto characters with many different drawings (the source of real cross references).

In [ ]:
from pathlib import Path

from scripts.make_splits import scan_pairs, is_aug

groups = scan_pairs(Path(DATA))
ad = {g: s for g, s in groups.items() if g.startswith("AnimeDiffusion_")}
chars = {g: s for g, s in groups.items() if not g.startswith("AnimeDiffusion_")}
print(f"pairs: {sum(map(len, groups.values()))} in {len(groups)} groups")
print(f"AnimeDiffusion groups: {len(ad)} ({sum(map(len, ad.values()))} images, {sum(not is_aug(x) for s in ad.values() for x in s)} originals)")
print(f"moeimouto characters: {len(chars)} ({sum(map(len, chars.values()))} images)")
sizes = sorted(len(s) for s in chars.values())
print("drawings per character: min", sizes[0], "median", sizes[len(sizes) // 2], "max", sizes[-1])
print("characters with > 15 drawings (eligible for test_cross):", sum(n > 15 for n in sizes))

## 3. Splits and pair files
Defaults: test = 579 AnimeDiffusion originals, val = 1,000, test_cross = 40 held-out characters x 15
drawings (600 sketches, none of these characters in training). Adjust `--cross-chars` / `--cross-per-char`
only if section 2 shows too few eligible characters, and tell the team.

If the official AnimeDiffusion test list has arrived, add `--test-ids <file>`.

In [ ]:
!python -m scripts.make_splits --root $DATA --out splits
!python -m scripts.make_pairs --root $DATA --splits-dir splits
!wc -l splits/*.txt splits/*.csv
!mkdir -p /kaggle/working/splits && cp splits/* /kaggle/working/splits/

## 4. Look at the loader output
Rows: sketch (after the shared clean-up) / target / reference.

In [ ]:
from IPython.display import Image, display
for split, mode in [("train", "train"), ("test", "self_ref"), ("test_cross", "cross_ref"), ("test", "shuffled_ref")]:
    out = f"/kaggle/working/previews/{split}_{mode}.png"
    !python -m scripts.preview_dataset --split $split --mode $mode --n 12 --out $out
    display(Image(out))

## 5. Next steps
1. Save version → create a private Kaggle Dataset from this notebook's output; share it with the team.
2. Download `/kaggle/working/splits/*`, put them in the repo's `splits/`, commit, open a PR.
3. In every other notebook: attach the dataset and set
   `os.environ["ANIME_DATA_ROOT"] = "/kaggle/input/<dataset>/AnimeFaceColorization"`.